# chess-vi — SFT trên Colab

Notebook này **chỉ gọi script** `chessvi.train.sft`. Không copy-paste logic vào cell:
logic nằm trong repo để test được và để Colab với local chạy đúng một thứ.

Runtime cần: **A100 / L4 / T4 GPU**. Colab hay ngắt giữa chừng nên script bật
`hub_strategy="checkpoint"` — cell cuối resume lại từ checkpoint.

Thứ tự chạy: `Mount` → `Cài đặt` → `Token` → `GPU` → `Validate` → `Smoke test` → `Train`.
Đứt kết nối thì chạy lại 4 cell đầu rồi nhảy thẳng xuống cell **Resume**.

## 1. Mount Google Drive

In [2]:
from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/chessvi"
!mkdir -p {DRIVE_ROOT}/outputs

Mounted at /content/drive


## 2. Lấy repo và cài đặt

Sửa `REPO_URL` thành remote của bạn. Nếu đã clone repo vào Drive thì chỉ cần `cd`.

In [13]:
REPO_URL = "https://github.com/trantrien1/ChessVi.git"
REPO_DIR = "/content/ChessVi"

import os

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull --ff-only || true

# Bỏ -q: -q nuốt mất chi tiết xung đột dependency, chỉ còn trơ
# "ResolutionImpossible" không biết package nào đá nhau.
!pip install -e ".[train,data]" 2>&1 | tail -30

# pip thất bại KHÔNG làm notebook dừng lại — không chốt ở đây thì mọi cell sau
# sẽ chết vì ModuleNotFoundError và che mất nguyên nhân thật.
!python -c "import chessvi; print('chessvi OK:', chessvi.__file__)" || echo ">>> CÀI ĐẶT HỎNG — DỪNG LẠI, ĐỪNG CHẠY CELL SAU"


/content/ChessVi
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 8 (delta 6), reused 8 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 2.89 KiB | 1.45 MiB/s, done.
From https://github.com/trantrien1/ChessVi
   ec729e7..a7bec47  main       -> origin/main
Updating ec729e7..a7bec47
Fast-forward
 src/chessvi/train/sft.py | 31 +++++++++++++++++++++++++++
 tests/test_sft.py        | 56 ++++++++++++++++++++++++++++++++++++++++++++++++
 2 files changed, 87 insertions(+)
  Building editable for chessvi (pyproject.toml): started
  Building editable for chessvi (pyproject.toml): finished with status 'done'
  Created wheel for chessvi: filename=chessvi-0.1.0-py3-none-any.whl size=4073 sha256=56018103bc6bd4da704ae86d57b2f3ba3f9d16728d7243f41bb1c6d48fb54153
  Stored in directory: /tmp/pip-ephem-wheel-cache-xk_rsym0/wheels/d3/81/62/f713bf3a1054567999022cdbc312cb8a1724e4566cc778afc9
Succe

## 3. Token Hugging Face

Lưu token trong **Colab Secrets** (biểu tượng chìa khoá, tên `HF_TOKEN`).
Không bao giờ dán token thẳng vào cell — notebook sẽ bị commit kèm token.

In [12]:
import os

# Colab Secrets chỉ đọc được khi chạy từ giao diện web colab.research.google.com:
# userdata.get() hỏi ngược về frontend trong trình duyệt. Chạy từ VS Code hay một
# frontend khác thì không ai trả lời -> TimeoutException. Bắt rộng vì mỗi trường
# hợp hỏng một kiểu (ImportError, TimeoutException, SecretNotFoundError).
if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata

        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
    except Exception as error:
        print(f"Không lấy được Colab Secret ({type(error).__name__}).")
        import getpass

        # getpass không in token ra output, nên notebook commit lên không lộ.
        os.environ["HF_TOKEN"] = getpass.getpass("HF_TOKEN (Enter để bỏ qua): ")

# Token CHỈ cần khi push lên Hub (cell 7). Các cell validate/smoke test không cần.
if not os.environ.get("HF_TOKEN"):
    print("CHƯA CÓ HF_TOKEN — validate và smoke test vẫn chạy, cell 7 sẽ không push được.")

In [15]:
import os, getpass
os.environ["HF_TOKEN"] = getpass.getpass("HF_TOKEN mới: ")

from huggingface_hub import HfApi
info = HfApi(token=os.environ["HF_TOKEN"]).whoami()
print(info["name"], info.get("auth", {}).get("accessToken", {}).get("role"))


anonymos111 write


## 4. Kiểm tra GPU

In [4]:
!nvidia-smi

import torch

print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")

Sat Sep 12 07:23:56 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             42W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 5. Validate dữ liệu đã dịch (T6)

`--out-clean` ghi tập đã pass ra `data/validated/sft` — đây chính là đầu vào
của SFT. Không có bước này thì không có file nào chứa "dữ liệu đã validate".

Kỳ vọng loại 5–15%. Trên 25% script sẽ in cảnh báo to: pipeline dịch có vấn đề,
quay lại T5 chứ đừng train.

**Chốt chặn thủ công:** trước khi sang cell tiếp theo, tự đọc tay 200 mẫu ngẫu
nhiên đã pass. Không ai thay bạn làm bước này được.

In [8]:
RAW = f"{DRIVE_ROOT}/data/raw/c1_sft.jsonl"
TRANSLATED = f"{DRIVE_ROOT}/data/translated/sft"

# Backend llm chạy bằng vLLM — chưa cài ở cell 2 nên cài ở đây.
!pip install -e ".[rl]" 2>&1 | tail -5

# Chuẩn hoá C1-data -> {id, fen, question, answer, label}. --limit để thử trước.
!python -m chessvi.data.c1 --split sft --limit 200 --out {RAW}

# Nhìn 20 cặp before/after. Ký hiệu cờ PHẢI còn nguyên sau khi dịch.
!python -m chessvi.data.translate --input-jsonl {RAW} --split sft --backend llm --limit 20 --dry-run


  Attempting uninstall: chessvi
    Found existing installation: chessvi 0.1.0
    Uninstalling chessvi-0.1.0:
      Successfully uninstalled chessvi-0.1.0
Streaming UofTCSSLab/C1-data config=sft split=train
Tổng số row đọc                                200
Chuẩn hoá được                                 200
Bỏ qua                                           0
Ghi 200 mẫu vào /content/drive/MyDrive/chessvi/data/raw/c1_sft.jsonl
Nạp Qwen/Qwen3-30B-A3B qua vLLM
INFO 09-12 03:13:24 [api_utils.py:272] non-default args: {'max_model_len': 5120, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-30B-A3B'}
INFO 09-12 03:13:26 [model.py:672] Resolved architecture: Qwen3MoeForCausalLM
INFO 09-12 03:13:26 [model.py:1965] Using max model len 5120
INFO 09-12 03:13:26 [scheduler.py:242] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 09-12 03:13:26 [kernel.py:308] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'],

In [10]:
!rm -f {DRIVE_ROOT}/data/raw/c1_sft.jsonl


In [11]:
import os
import time

RAW = f"{DRIVE_ROOT}/data/raw/c1_sft.jsonl"
TRANSLATED_DIR = f"{DRIVE_ROOT}/data/translated"
TRANSLATED = f"{TRANSLATED_DIR}/sft"

# vLLM nằm trong extra [rl]; cell 2 chỉ cài [train,data].
!pip install -e ".[rl]" 2>&1 | tail -5
!python -c "import vllm" || echo ">>> vLLM HỎNG — DỪNG LẠI, ĐỪNG CHẠY TIẾP"

# Chuẩn hoá đủ 39.601 mẫu. Bỏ qua nếu đã có: --resume bỏ mẫu theo THỨ TỰ DÒNG,
# nên file này phải y hệt nhau giữa các lần chạy.
if os.path.exists(RAW):
    print(f"Đã có {RAW} — bỏ qua bước chuẩn hoá")
else:
    !python -m chessvi.data.c1 --split sft --out {RAW}
!wc -l {RAW}

# Chạy thật, vài tiếng. Ghi thẳng vào Drive, checkpoint mỗi 500 mẫu.
start = time.time()
!python -m chessvi.data.translate --input-jsonl {RAW} --split sft --backend llm --batch-size 256 --out-dir {TRANSLATED_DIR} --resume
print(f"Dịch xong sau {(time.time() - start) / 3600:.2f} giờ")

# Xác nhận dữ liệu nằm trên Drive chứ không phải /content (mất khi runtime chết).
!ls -la {TRANSLATED}
!cat {TRANSLATED}/_state.json


  Attempting uninstall: chessvi
    Found existing installation: chessvi 0.1.0
    Uninstalling chessvi-0.1.0:
      Successfully uninstalled chessvi-0.1.0
Streaming UofTCSSLab/C1-data config=sft split=train
Tổng số row đọc                              39601
Chuẩn hoá được                               39601
Bỏ qua                                           0
Ghi 39601 mẫu vào /content/drive/MyDrive/chessvi/data/raw/c1_sft.jsonl
39601 /content/drive/MyDrive/chessvi/data/raw/c1_sft.jsonl
Nạp Qwen/Qwen3-30B-A3B qua vLLM
INFO 09-12 03:24:54 [api_utils.py:272] non-default args: {'max_model_len': 5120, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-30B-A3B'}
INFO 09-12 03:24:56 [model.py:672] Resolved architecture: Qwen3MoeForCausalLM
INFO 09-12 03:24:56 [model.py:1965] Using max model len 5120
INFO 09-12 03:24:56 [scheduler.py:242] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 09-12 03:24:56 [kernel.py:308] Final IR op priority after setti

In [5]:
TRANSLATED = f"{DRIVE_ROOT}/data/translated/sft"
DATA = f"{DRIVE_ROOT}/data/validated/sft"

!python -m chessvi.data.validate \
    --input {TRANSLATED} \
    --rejected {DRIVE_ROOT}/data/rejected.jsonl \
    --out-clean {DATA}

!ls -la {DATA}


Ghi 39355 mẫu đã pass vào /content/drive/MyDrive/chessvi/data/validated/sft
Tổng số mẫu                            39601
Pass                                   39355      99.38%
Loại                                     246       0.62%
--------------------------------------------------------
Lý do loại (một mẫu có thể nhiều lý do)
  english_chunk                          246       0.62%
Mẫu bị loại đã ghi ra /content/drive/MyDrive/chessvi/data/rejected.jsonl
total 16977
drwx------ 2 root root     4096 Sep 12 07:28 .
drwx------ 3 root root     4096 Sep 12 05:22 ..
-rw------- 1 root root 17375958 Sep 12 07:28 part-00000.parquet


## 6. Smoke test

5 step với model 0.6B. Cell này phải chạy xong không lỗi **trước khi** tốn CU
cho lần train thật.

In [6]:
DATA = f"{DRIVE_ROOT}/data/validated/sft"

!python -m chessvi.train.sft \
    --data {DATA} \
    --base-model Qwen/Qwen3-0.6B \
    --limit 50 --max-steps 5 \
    --output-dir /content/outputs/smoke \
    --no-push

2026-09-12 07:33:35,373 INFO Nạp 50 mẫu từ /content/drive/MyDrive/chessvi/data/validated/sft
config.json: 100% 726/726 [00:00<00:00, 3.05MB/s]
tokenizer_config.json: 100% 9.73k/9.73k [00:00<00:00, 5.85MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 13.0MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 13.1MB/s]

tokenizer.json: downloading bytes:   0% 0.00/11.4M [00:00<?, ?B/s]s]
tokenizer.json: downloading bytes: 100% 3.40M/3.40M [00:01<00:00, 2.85MB/s,  329kB/s  ]
tokenizer.json: reconstructing file: 100% 11.4M/11.4M [00:01<00:00, 9.57MB/s, 1.11MB/s  ]
2026-09-12 07:33:43,015 INFO Tokenize xong 50 mẫu (max_length=2048)
2026-09-12 07:33:43,032 INFO Nạp base model Qwen/Qwen3-0.6B (4-bit: True)

model.safetensors: downloading bytes:   4% 65.7M/1.50G [00:01<00:23, 61.8MB/s, 2.16MB/s  ]
model.safetensors: downloading bytes:  13% 197M/1.50G [00:01<00:05, 221MB/s, 12.5MB/s  ]  
model.safetensors: downloading bytes:  19% 292M/1.50G [00:01<00:04, 298MB/s, 22.5MB/s  ]s  ]
model.safetensors: do

## 7. Train thật

Đổi `HUB_MODEL_ID` thành repo của bạn. Repo được tạo ở chế độ **private**.

In [17]:
DATA = f"{DRIVE_ROOT}/data/validated/sft"

HUB_MODEL_ID = "anonymos111/chessvi-4b-sft"
OUTPUT_DIR = f"{DRIVE_ROOT}/outputs/sft"

!python -m chessvi.train.sft \
    --data {DATA} \
    --base-model Qwen/Qwen3-4B \
    --output-dir {OUTPUT_DIR} \
    --hub-model-id {HUB_MODEL_ID} \
    --epochs 2 --batch-size 16 --grad-accum 16 --save-steps 200

2026-09-12 07:58:23,304 INFO Repo Hub anonymos111/chessvi-4b-sft sẵn sàng
2026-09-12 07:58:24,194 INFO Nạp 39355 mẫu từ /content/drive/MyDrive/chessvi/data/validated/sft
2026-09-12 07:59:34,097 INFO Tokenize xong 39355 mẫu (max_length=2048)
2026-09-12 07:59:38,354 INFO Nạp base model Qwen/Qwen3-4B (4-bit: True)
Loading weights: 100% 398/398 [00:02<00:00, 168.44it/s]
trainable params: 66,060,288 || all params: 4,088,528,384 || trainable%: 1.6157
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
{'loss': '2.787', 'grad_norm': '0.9236', 'learning_rate': '0.00018', 'epoch': '0.06504'}
{'loss': '1.79', 'grad_norm': '0.3292', 'learning_rate': '0.0001996', 'epoch': '0.1301'}
{'loss': '1.508', 'grad_norm': '0.273', 'learning_rate': '0.000198', 'epoch': '0.1951'}
{'loss': '1.378', 'grad_norm': '0.2898', 'learning_rate': '0.0001954', 'epoch': '0.2602'}
{'loss': '1.286', 'grad_norm': '0.3321', 'learning_rate': '0.0001917', 'epoch': '0.3252'}
{'loss': '1.226', 'gr

In [ ]:
OUTPUT_DIR = f"{DRIVE_ROOT}/outputs/sft"
PUZZLES = f"{DRIVE_ROOT}/data/puzzles/test.parquet"
REPORT = f"{DRIVE_ROOT}/reports/sft_acc.csv"

!python -m chessvi.eval.puzzle_acc \
    --puzzles {PUZZLES} \
    --backend hf \
    --model-path Qwen/Qwen3-4B \
    --adapter {OUTPUT_DIR} \
    --model-name chessvi-4b-sft \
    --limit 300 \
    --out {REPORT}


In [18]:
DATA = f"{DRIVE_ROOT}/data/validated/sft"

HUB_MODEL_ID_30B = "anonymos111/chessvi-30b-sft"
OUTPUT_DIR_30B = f"{DRIVE_ROOT}/outputs/sft-30b"

!python -m chessvi.train.sft \
    --data {DATA} \
    --base-model Qwen/Qwen3-30B-A3B \
    --output-dir {OUTPUT_DIR_30B} \
    --hub-model-id {HUB_MODEL_ID_30B} \
    --epochs 2 --batch-size 8 --grad-accum 4 --save-steps 200


2026-09-12 11:29:49,695 INFO Repo Hub anonymos111/chessvi-30b-sft sẵn sàng
2026-09-12 11:29:50,577 INFO Nạp 39355 mẫu từ /content/drive/MyDrive/chessvi/data/validated/sft
config.json: 100% 963/963 [00:00<00:00, 3.71MB/s]
tokenizer_config.json: 100% 9.73k/9.73k [00:00<00:00, 20.7MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 13.0MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 6.50MB/s]

tokenizer.json: downloading bytes:   0% 0.00/11.4M [00:00<?, ?B/s]s]
tokenizer.json: downloading bytes:  29% 3.28M/11.4M [00:01<00:02, 3.03MB/s,  124kB/s  ]
tokenizer.json: downloading bytes: 100% 3.40M/3.40M [00:01<00:00, 2.54MB/s,  327kB/s  ] ]
tokenizer.json: reconstructing file: 100% 11.4M/11.4M [00:01<00:00, 8.54MB/s, 1.10MB/s  ]
2026-09-12 11:31:06,472 INFO Tokenize xong 39355 mẫu (max_length=2048)
2026-09-12 11:31:10,734 INFO Nạp base model Qwen/Qwen3-30B-A3B (4-bit: True)
model.safetensors.index.json: 100% 1.70M/1.70M [00:00<00:00, 47.8MB/s]
Reconstructing (incomplete total...): |          |  

## 8. Resume sau khi Colab ngắt

Chạy lại cell 1–4 rồi chạy cell này. `--resume` đọc checkpoint mới nhất trong
`--output-dir`; nếu output nằm trên Drive thì checkpoint vẫn còn nguyên.

In [ ]:
DATA = f"{DRIVE_ROOT}/data/validated/sft"
HUB_MODEL_ID = "anonymos111/chessvi-4b-sft"
OUTPUT_DIR = f"{DRIVE_ROOT}/outputs/sft"

!ls -la {OUTPUT_DIR} | head -20

!python -m chessvi.train.sft \
    --data {DATA} \
    --base-model Qwen/Qwen3-4B \
    --output-dir {OUTPUT_DIR} \
    --hub-model-id {HUB_MODEL_ID} \
    --epochs 2 --batch-size 1 --grad-accum 16 --save-steps 200 \
    --resume

## 9. Chốt chặn sau T8

Accuracy trên test set phải đạt **30–38%**. Dưới 20% nghĩa là dữ liệu có vấn đề
— quay lại T5, **đừng** chạy RL.

```bash
python -m chessvi.eval.puzzle_acc --puzzles data/puzzles/test.parquet \
    --backend hf --model-path outputs/sft --out reports/sft_acc.csv
```